In [ ]:
import pandas as pd
import numpy as np
bcg = pd.read_csv("03_product_bcg_line_clean_line_fixed.csv")
bcg.head()

,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant
0,CITYBIKE_P,Xe phổ thông,GN 05 24,0.000000e+00,2.200000e+06,1.000000,0.000027,Question Mark
1,CITYBIKE_P,Xe phổ thông,GN 06 - 27 2.0 Pro Shimano,0.000000e+00,3.416667e+06,1.000000,0.000042,Question Mark
2,CITYBIKE_P,Xe phổ thông,GN 06 D,0.000000e+00,6.722222e+08,1.000000,0.008266,Question Mark
3,CITYBIKE_P,Xe phổ thông,LD 24,1.808562e+09,4.225101e+09,1.336166,0.051951,Star
4,CITYBIKE_P,Xe phổ thông,LD 26,9.717500e+06,7.766444e+09,798.222409,0.095495,Star


In [ ]:
# =========================
# 2. Clean numeric columns
# =========================

numeric_cols = [
    "revenue_q1_2025",
    "revenue_q1_2026",
    "growth_rate",
    "revenue_share"
]

for col in numeric_cols:
    if col in bcg.columns:
        bcg[col] = (
            bcg[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .replace(["nan", "None", "NULL", ""], np.nan)
        )
        bcg[col] = pd.to_numeric(bcg[col], errors="coerce")

print(bcg.dtypes)
display(bcg.head())

group_code          object
group_name          object
line_name           object
revenue_q1_2025    float64
revenue_q1_2026    float64
growth_rate        float64
revenue_share      float64
bcg_quadrant        object
dtype: object


,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant
0,CITYBIKE_P,Xe phổ thông,GN 05 24,0.000000e+00,2.200000e+06,1.000000,0.000027,Question Mark
1,CITYBIKE_P,Xe phổ thông,GN 06 - 27 2.0 Pro Shimano,0.000000e+00,3.416667e+06,1.000000,0.000042,Question Mark
2,CITYBIKE_P,Xe phổ thông,GN 06 D,0.000000e+00,6.722222e+08,1.000000,0.008266,Question Mark
3,CITYBIKE_P,Xe phổ thông,LD 24,1.808562e+09,4.225101e+09,1.336166,0.051951,Star
4,CITYBIKE_P,Xe phổ thông,LD 26,9.717500e+06,7.766444e+09,798.222409,0.095495,Star


In [ ]:
# =========================
# 3. Current BCG distribution
# =========================

print("Current BCG quadrant distribution:")

quadrant_summary = (
    bcg.groupby("bcg_quadrant", dropna=False)
    .agg(
        num_lines=("line_name", "count"),
        total_revenue_q1_2025=("revenue_q1_2025", "sum"),
        total_revenue_q1_2026=("revenue_q1_2026", "sum"),
        avg_growth_rate=("growth_rate", "mean"),
        avg_revenue_share=("revenue_share", "mean")
    )
    .reset_index()
    .sort_values("total_revenue_q1_2026", ascending=False)
)

display(quadrant_summary)

Current BCG quadrant distribution:


,bcg_quadrant,num_lines,total_revenue_q1_2025,total_revenue_q1_2026,avg_growth_rate,avg_revenue_share
2,Star,7,1.595211e+10,4.680717e+10,115.676949,0.082219
1,Question Mark,47,8.953657e+09,3.326913e+10,1.136543,0.008704
0,Dog,17,3.211404e+09,1.251689e+09,-0.732448,0.000905


In [ ]:
# =========================
# 4. Recalculate BCG quadrant
# =========================

valid = bcg[
    bcg["growth_rate"].notna() &
    bcg["revenue_share"].notna()
].copy()

growth_threshold = valid["growth_rate"].median()
share_threshold = valid["revenue_share"].median()

print("Growth threshold:", growth_threshold)
print("Revenue share threshold:", share_threshold)

def assign_bcg(row):
    high_share = row["revenue_share"] >= share_threshold
    high_growth = row["growth_rate"] >= growth_threshold

    if high_share and high_growth:
        return "Star"
    elif high_share and not high_growth:
        return "Cash Cow"
    elif not high_share and high_growth:
        return "Question Mark"
    else:
        return "Dog"

bcg["bcg_recalculated"] = bcg.apply(assign_bcg, axis=1)

display(
    bcg.groupby("bcg_recalculated")
    .agg(
        num_lines=("line_name", "count"),
        total_revenue_q1_2026=("revenue_q1_2026", "sum"),
        avg_growth_rate=("growth_rate", "mean"),
        avg_revenue_share=("revenue_share", "mean")
    )
    .reset_index()
    .sort_values("total_revenue_q1_2026", ascending=False)
)

Growth threshold: 1.0
Revenue share threshold: 0.003501840613966


,bcg_recalculated,num_lines,total_revenue_q1_2026,avg_growth_rate,avg_revenue_share
3,Star,29,6.717428e+10,29.073773,0.028482
0,Cash Cow,7,1.154005e+10,0.557183,0.020271
1,Dog,22,1.346605e+09,-0.452010,0.000753
2,Question Mark,13,1.267059e+09,1.046852,0.001198


In [ ]:


cash_cows = bcg[bcg["bcg_recalculated"] == "Cash Cow"].copy()

cash_cows = cash_cows.sort_values("revenue_q1_2026", ascending=False)

print("Number of recalculated Cash Cows:", len(cash_cows))

display(
    cash_cows[
        [
            "group_code",
            "group_name",
            "line_name",
            "revenue_q1_2025",
            "revenue_q1_2026",
            "growth_rate",
            "revenue_share",
            "bcg_quadrant",
            "bcg_recalculated"
        ]
    ]
)

Number of recalculated Cash Cows: 7


,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant,bcg_recalculated
58,SPORTBIKE_STEEL,Xe thể thao thép,MTB SPD 27.5,4.423463e+09,7.733950e+09,0.748392,0.095096,Star,Cash Cow
9,CITYBIKE_P,Xe phổ thông,Xe 219 - 24,4.402907e+08,8.076637e+08,0.834387,0.009931,Question Mark,Cash Cow
37,KIDBIKE,Xe trẻ em nhóm 1,Xe TE Super 20,4.295469e+08,7.806212e+08,0.817313,0.009598,Question Mark,Cash Cow
8,CITYBIKE_P,Xe phổ thông,Xe 219 - 05 - 26,7.139514e+08,7.717414e+08,0.080944,0.009489,Dog,Cash Cow
29,KIDBIKE,Xe trẻ em,Neo 20 - 03,5.588936e+08,7.001780e+08,0.252793,0.008609,Question Mark,Cash Cow
46,KIDBIKE,Xe trẻ em nhóm 2,Xe TE Batman16,2.378870e+08,4.430494e+08,0.862436,0.005448,Question Mark,Cash Cow
12,CITYBIKE_P,Xe phổ thông,Xe GN 05 - 27,2.322410e+08,3.028462e+08,0.304017,0.003724,Question Mark,Cash Cow


In [ ]:
# =========================
# 6. Compare original vs recalculated quadrant
# =========================

changed_quadrants = bcg[
    bcg["bcg_quadrant"].astype(str).str.strip()
    != bcg["bcg_recalculated"].astype(str).str.strip()
].copy()

changed_quadrants = changed_quadrants.sort_values(
    ["bcg_recalculated", "revenue_q1_2026"],
    ascending=[True, False]
)

print("Number of changed quadrant rows:", len(changed_quadrants))

display(
    changed_quadrants[
        [
            "group_code",
            "group_name",
            "line_name",
            "revenue_q1_2025",
            "revenue_q1_2026",
            "growth_rate",
            "revenue_share",
            "bcg_quadrant",
            "bcg_recalculated"
        ]
    ]
)

Number of changed quadrant rows: 36


,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant,bcg_recalculated
58,SPORTBIKE_STEEL,Xe thể thao thép,MTB SPD 27.5,4.423463e+09,7.733950e+09,0.748392,0.095096,Star,Cash Cow
9,CITYBIKE_P,Xe phổ thông,Xe 219 - 24,4.402907e+08,8.076637e+08,0.834387,0.009931,Question Mark,Cash Cow
37,KIDBIKE,Xe trẻ em nhóm 1,Xe TE Super 20,4.295469e+08,7.806212e+08,0.817313,0.009598,Question Mark,Cash Cow
8,CITYBIKE_P,Xe phổ thông,Xe 219 - 05 - 26,7.139514e+08,7.717414e+08,0.080944,0.009489,Dog,Cash Cow
29,KIDBIKE,Xe trẻ em,Neo 20 - 03,5.588936e+08,7.001780e+08,0.252793,0.008609,Question Mark,Cash Cow
46,KIDBIKE,Xe trẻ em nhóm 2,Xe TE Batman16,2.378870e+08,4.430494e+08,0.862436,0.005448,Question Mark,Cash Cow
12,CITYBIKE_P,Xe phổ thông,Xe GN 05 - 27,2.322410e+08,3.028462e+08,0.304017,0.003724,Question Mark,Cash Cow
53,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX AT 27.5 2.0,1.578444e+08,2.343600e+08,0.484753,0.002882,Question Mark,Dog
66,SPORTBIKE_STEEL,Xe thể thao thép,Xe Highway 27.5,1.396616e+08,1.594200e+08,0.141474,0.001960,Question Mark,Dog
40,KIDBIKE,Xe trẻ em nhóm 2,Xe Love 12,9.281620e+07,1.573611e+08,0.695405,0.001935,Question Mark,Dog


In [ ]:
# =========================
# 7. High revenue share + low growth candidates
# =========================

cash_cow_candidates = bcg[
    (bcg["revenue_share"] >= share_threshold) &
    (bcg["growth_rate"] < growth_threshold)
].copy()

cash_cow_candidates = cash_cow_candidates.sort_values(
    "revenue_q1_2026",
    ascending=False
)

display(
    cash_cow_candidates[
        [
            "group_code",
            "group_name",
            "line_name",
            "revenue_q1_2025",
            "revenue_q1_2026",
            "growth_rate",
            "revenue_share",
            "bcg_quadrant",
            "bcg_recalculated"
        ]
    ]
)

,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant,bcg_recalculated
58,SPORTBIKE_STEEL,Xe thể thao thép,MTB SPD 27.5,4.423463e+09,7.733950e+09,0.748392,0.095096,Star,Cash Cow
9,CITYBIKE_P,Xe phổ thông,Xe 219 - 24,4.402907e+08,8.076637e+08,0.834387,0.009931,Question Mark,Cash Cow
37,KIDBIKE,Xe trẻ em nhóm 1,Xe TE Super 20,4.295469e+08,7.806212e+08,0.817313,0.009598,Question Mark,Cash Cow
8,CITYBIKE_P,Xe phổ thông,Xe 219 - 05 - 26,7.139514e+08,7.717414e+08,0.080944,0.009489,Dog,Cash Cow
29,KIDBIKE,Xe trẻ em,Neo 20 - 03,5.588936e+08,7.001780e+08,0.252793,0.008609,Question Mark,Cash Cow
46,KIDBIKE,Xe trẻ em nhóm 2,Xe TE Batman16,2.378870e+08,4.430494e+08,0.862436,0.005448,Question Mark,Cash Cow
12,CITYBIKE_P,Xe phổ thông,Xe GN 05 - 27,2.322410e+08,3.028462e+08,0.304017,0.003724,Question Mark,Cash Cow


In [ ]:
# =========================
# 8. Lines with zero 2025 revenue
# =========================

zero_2025 = bcg[
    (bcg["revenue_q1_2025"].isna()) |
    (bcg["revenue_q1_2025"] == 0)
].copy()

zero_2025 = zero_2025.sort_values("revenue_q1_2026", ascending=False)

print("Number of lines with zero or missing Q1 2025 revenue:", len(zero_2025))

display(
    zero_2025[
        [
            "group_code",
            "group_name",
            "line_name",
            "revenue_q1_2025",
            "revenue_q1_2026",
            "growth_rate",
            "revenue_share",
            "bcg_quadrant",
            "bcg_recalculated"
        ]
    ]
)

Number of lines with zero or missing Q1 2025 revenue: 22


,group_code,group_name,line_name,revenue_q1_2025,revenue_q1_2026,growth_rate,revenue_share,bcg_quadrant,bcg_recalculated
7,CITYBIKE_P,Xe phổ thông,SK 24,0.0,2.341072e+09,1.0,0.028786,Question Mark,Star
60,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,0.0,2.180740e+09,1.0,0.026814,Question Mark,Star
6,CITYBIKE_P,Xe phổ thông,SK 20,0.0,1.786432e+09,1.0,0.021966,Question Mark,Star
28,KIDBIKE,Xe trẻ em,LD 26,0.0,9.108583e+08,1.0,0.011200,Question Mark,Star
5,CITYBIKE_P,Xe phổ thông,MS 27.5,0.0,7.489314e+08,1.0,0.009209,Question Mark,Star
2,CITYBIKE_P,Xe phổ thông,GN 06 D,0.0,6.722222e+08,1.0,0.008266,Question Mark,Star
34,KIDBIKE,Xe trẻ em nhóm 1,Xe GN 06 2.0 20 DC,0.0,5.617214e+08,1.0,0.006907,Question Mark,Star
26,KIDBIKE,Xe trẻ em,Batwheels 16,0.0,4.924574e+08,1.0,0.006055,Question Mark,Star
27,KIDBIKE,Xe trẻ em,Bubbles 20,0.0,2.881227e+08,1.0,0.003543,Question Mark,Star
52,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX 2.0,0.0,2.847977e+08,1.0,0.003502,Question Mark,Star


In [ ]:


bcg_checked = bcg.copy()

output_file = "product_bcg_checked_cash_cows.csv"
bcg_checked.to_csv(output_file, index=False, encoding="utf-8-sig")
